# 01. Formal Exploratory Data Analysis

Structured deep-dive into the dataset. Builds on the preliminary EDA with richer diagnostics.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
%matplotlib inline

In [ ]:
raw_dir = Path.cwd().parent / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

csv_files = list(raw_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Place a CSV in {raw_dir}")

file_path = csv_files[0]
df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
print(f"Loaded: {file_path.name}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## 1. Schema Audit

Complete column-by-column diagnostic: type, nulls, cardinality, memory.

In [ ]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "cardinality": df.nunique(dropna=False),
    "memory_kb": (df.memory_usage(deep=True) / 1024).round(1)
})
display(schema)
print(f"Total memory: {schema['memory_kb'].sum():,.1f} KB")

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Numerical: {len(num_cols)}  |  Categorical: {len(cat_cols)}")

## 2. Missing Values - Full Diagnosis

In [ ]:
missing_cols = schema[schema["null_pct"] > 0][["null_pct"]].sort_values("null_pct", ascending=False)
if not missing_cols.empty:
    display(missing_cols)
else:
    print("No missing values.")

In [ ]:
nulls_per_row = df.isna().sum(axis=1)
print("Rows with >=X nulls:")
for t in [1, 2, 5, 10, 20, 50]:
    n = (nulls_per_row >= t).sum()
    print(f"  >={t:>3}: {n:>6,} ({n/len(df)*100:>5.1f}%)")

if nulls_per_row.max() > 0:
    plt.figure(figsize=(10, 3))
    sns.histplot(nulls_per_row, bins=min(50, nulls_per_row.max()+1), discrete=True)
    plt.title("Nulls per Row")
    plt.tight_layout()
    plt.show()

In [ ]:
null_pct = df.isna().mean() * 100
rec_df = pd.DataFrame({
    "null_pct": null_pct.round(2),
    "decision": null_pct.apply(lambda p: "DROP (>40%)" if p > 40 else "KEEP")
}).sort_values("null_pct", ascending=False)

to_drop = rec_df[rec_df["decision"] == "DROP (>40%)"]
n_drop = len(to_drop)
print(f"Columns recommended for removal (null >40%): {n_drop}")
if n_drop:
    display(to_drop)

## 3. Numerical Analysis

In [ ]:
if num_cols:
    stats = df[num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
    stats["skew"] = df[num_cols].skew().round(3)
    stats["kurtosis"] = df[num_cols].kurtosis().round(3)
    display(stats)

In [ ]:
if num_cols:
    n_hist = min(9, len(num_cols))
    rows = (n_hist + 2) // 3
    fig, axes = plt.subplots(rows, 3, figsize=(14, 4 * rows))
    axes = axes.flatten()
    for ax, col in zip(axes, num_cols[:n_hist]):
        sns.histplot(df[col].dropna(), kde=True, bins=40, ax=ax)
        ax.set_title(col)
    for ax in axes[n_hist:]:
        ax.set_visible(False)
    plt.tight_layout()
    plt.show()

## 4. Categorical Analysis

In [ ]:
if cat_cols:
    cat_summary = pd.DataFrame({
        "unique": df[cat_cols].nunique(),
        "top_val": [df[c].value_counts(dropna=False).index[0] if df[c].nunique() > 0 else None for c in cat_cols],
        "top_pct": [round(df[c].value_counts(dropna=False).iloc[0] / len(df) * 100, 1) for c in cat_cols]
    })
    display(cat_summary)

In [ ]:
for col in cat_cols[:6]:
    vc = df[col].value_counts(dropna=False).head(12)
    plt.figure(figsize=(8, 3))
    sns.barplot(x=vc.values, y=vc.index, hue=vc.index, palette="viridis", legend=False)
    plt.title(f"{col} (top 12)")
    plt.xlabel("Count")
    plt.tight_layout()
    plt.show()

## 5. Correlation Analysis

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    plt.figure(figsize=(max(8, len(num_cols) * 0.6), max(6, len(num_cols) * 0.5)))
    sns.heatmap(corr, mask=mask, annot=len(num_cols) <= 15,
                fmt=".2f", cmap="RdBu_r", center=0,
                square=True, linewidths=0.5)
    plt.title("Correlation Matrix")
    plt.tight_layout()
    plt.show()

    corr_u = corr.unstack()
    corr_u = corr_u[corr_u.index.get_level_values(0) != corr_u.index.get_level_values(1)]
    top = corr_u.abs().sort_values(ascending=False).drop_duplicates().head(10)
    print("Top 10 strongest correlations:")
    for (c1, c2), v in top.items():
        print(f"  {c1:25s} vs {c2:25s}: {corr.loc[c1, c2]:+.4f}")

## 6. Outlier Analysis

In [ ]:
if num_cols:
    rows = []
    for col in num_cols:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        out = df[(df[col] < low) | (df[col] > high)][col].dropna()
        if len(out):
            rows.append({"col": col, "n": len(out), "%": round(len(out)/df[col].notna().sum()*100, 2)})
    out_df = pd.DataFrame(rows).sort_values("n", ascending=False)
    if not out_df.empty:
        display(out_df)
    else:
        print("No outliers detected.")
else:
    out_df = pd.DataFrame()

## 7. Summary & Recommendations

In [ ]:
print("=" * 60)
print("EDA SUMMARY")
print("=" * 60)
print(f"  Dataset:                {file_path.name}")
print(f"  Dimensions:             {df.shape[0]:>8,} rows x {df.shape[1]:,} cols")
print(f"  Missing cells:          {df.isna().sum().sum():>8,} ({df.isna().sum().sum()/df.size*100:.2f}%)")
print(f"  Rows with nulls:        {(df.isna().sum(axis=1)>=1).sum():>8,}")
print(f"  Duplicate rows:         {df.duplicated().sum():>8,}")
print(f"  Memory:                 {df.memory_usage(deep=True).sum()/1024**2:.2f} MB")
print(f"  Num cols:               {len(num_cols):>8}")
print(f"  Cat cols:               {len(cat_cols):>8}")
print(f"  Cols to drop (>40%):    {n_drop:>8}")
if num_cols:
    print(f"  Outlier cols:           {len(out_df):>8}")